# GBM Models: NASDAQ-100 Microstructure

With 114 symbols at 15-minute frequency, we have 13M+ training samples --- ideal
for GBM's appetite for data volume. The question is whether order-flow and
microstructure features contain non-linear predictive structure that tree-based
models can exploit beyond the linear baseline.

GBM modestly improves on the linear baseline (IC +0.007 vs +0.006), but the gain
is smaller than in daily case studies. At high frequency, the signal is predominantly
linear, and deeper tree configurations add limited value despite 13M samples.

**Learning Objectives**:
- Test whether non-linear models improve on the linear microstructure baseline
- Observe how tree depth interacts with signal-to-noise at high frequency
- Compare loss functions (MSE, MAE, Huber) on microstructure return distributions

**Book Reference**: Chapter 12, Section 12.2 (GBM Libraries)

**Prerequisites**: `03_financial_features.py`, `04_temporal.py`, [`05_evaluation`](05_evaluation.ipynb)

In [1]:
"""GBM Grid Search — config-driven regularization profiles × loss functions."""

import warnings

import numpy as np
import polars as pl
import yaml

from case_studies.utils.gbm import (
    prepare_gbm_folds,
    register_gbm_result,
    train_gbm_config,
)
from case_studies.utils.registry import (
    build_training_spec,
    get_training_dir,
    load_prediction_metrics,
    load_prediction_sets,
    training_hash_from_spec,
    training_run_status,
)
from utils.modeling import append_holdout_fold_if_needed, load_configs, load_modeling_dataset
from utils.paths import get_case_study_dir

warnings.filterwarnings("ignore")

In [2]:
CASE_STUDY_ID = "nasdaq100_microstructure"
PRIMARY_LABEL = ""
MAX_SYMBOLS = 0
MAX_FOLDS = 0
FORCE_RETRAIN = False  # Set True to retrain configs that already have complete hashes
PREDICTION_SPLIT = "validation"
TRAIN_SAMPLE_FRAC = 1.0  # <1.0 subsamples training rows per fold (val is never sampled). Use for memory-constrained runs on large datasets.

In [3]:
CASE_DIR = get_case_study_dir(CASE_STUDY_ID)
setup = yaml.safe_load((CASE_DIR / "config" / "setup.yaml").read_text())

if not PRIMARY_LABEL:
    PRIMARY_LABEL = setup["labels"]["primary"]

# Device: read from setup.yaml, fall back to GPU detection
gbm_config = setup.get("modeling", {}).get("gbm", {})
DEVICE = gbm_config.get("device", "cuda")
MAX_BIN = 63  # GPU default
import torch

if DEVICE != "cpu" and not torch.cuda.is_available():
    DEVICE, MAX_BIN = "cpu", 255

print(f"Case study: {CASE_STUDY_ID} | Device: {DEVICE} | max_bin: {MAX_BIN}")

Case study: nasdaq100_microstructure | Device: cpu | max_bin: 63


## 1. Load Data and Model Configs

GBM configs are defined in `config/training/{label}.yaml` under the `gbm:` key.
Each config references a preset in `config/lgb/` with the complete
LightGBM parameter set. To modify the grid, edit the label config file.

In [4]:
mds = load_modeling_dataset(CASE_STUDY_ID, PRIMARY_LABEL, max_symbols=MAX_SYMBOLS)
append_holdout_fold_if_needed(mds, PREDICTION_SPLIT, CASE_STUDY_ID)

dataset = mds.dataset
feature_names = mds.feature_names
label_col = mds.label_col
date_col = mds.date_col
entity_col = mds.entity_cols[0] if mds.entity_cols else "symbol"
splits = mds.splits[: MAX_FOLDS or None]

print(f"Dataset: {len(dataset):,} rows × {len(feature_names)} features")
print(f"Label: {label_col} | Task: {mds.task_type} | Folds: {len(splits)}")

Dataset: 16,032,651 rows × 88 features
Label: fwd_ret_15m | Task: regression | Folds: 2


In [5]:
configs = load_configs(CASE_STUDY_ID, PRIMARY_LABEL, family="gbm")

print(f"\n{len(configs)} configs × {len(splits)} folds\n")
for cfg in configs:
    leaves = cfg["params"].get("num_leaves", 31)
    obj = cfg["params"].get("objective", "regression")
    n_trees = cfg.get("max_iterations", 500)
    print(f"  {cfg['config_name']:25s}  leaves={leaves:3d}  obj={obj}  trees={n_trees}")


15 configs × 2 folds

  leaves_63_mae              leaves= 63  obj=regression_l1  trees=500
  leaves_7_mae               leaves=  7  obj=regression_l1  trees=500
  leaves_31_mae              leaves= 31  obj=regression_l1  trees=500
  leaves_31_huber            leaves= 31  obj=huber  trees=500
  leaves_15_huber            leaves= 15  obj=huber  trees=500
  leaves_63_huber            leaves= 63  obj=huber  trees=500
  leaves_15_mae              leaves= 15  obj=regression_l1  trees=500
  default_mae                leaves= 31  obj=regression_l1  trees=500
  leaves_7_huber             leaves=  7  obj=huber  trees=500
  default_huber              leaves= 31  obj=huber  trees=500
  default_mse                leaves= 31  obj=regression  trees=500
  leaves_7_mse               leaves=  7  obj=regression  trees=500
  leaves_31_mse              leaves= 31  obj=regression  trees=500
  leaves_15_mse              leaves= 15  obj=regression  trees=500
  leaves_63_mse              leaves= 63  obj=regr

## 2. Prepare CV Folds

GBM folds use float32 (LightGBM native precision) and skip
imputation/scaling — gradient boosting handles missing values natively.

In [6]:
dataset_pd = dataset.to_pandas()
fold_data = prepare_gbm_folds(
    dataset_pd,
    splits,
    feature_names,
    label_col,
    date_col,
    entity_col,
    task_type=mds.task_type,
    class_values=mds.class_values,
    temporal_by_fold=mds.temporal_by_fold,
    temporal_keys=mds.temporal_keys,
    temporal_feature_names=mds.temporal_feature_names,
    train_sample_frac=TRAIN_SAMPLE_FRAC,
)

for f in fold_data:
    print(f"  Fold {f['fold']}: train={f['n_train']:,}  val={f['n_val']:,}")

  Fold 0: train=4,032,199  val=4,018,166
  Fold 1: train=3,982,526  val=4,033,714


## 3. Train All Configs

For each config, train one LightGBM model per fold to `max_iterations` trees.
Cross-sectional IC is evaluated at checkpoints (every 50 iterations) to
detect overfitting — configs that peak early and decay indicate too much capacity.

In [7]:
results = []
for cfg in configs:
    # Pre-compute registry training dir so boosters go directly there
    spec = build_training_spec(
        cfg["family"],
        cfg["config_name"],
        label_col,
        n_folds=len(fold_data),
        max_bin=MAX_BIN,
        checkpoint_interval=cfg.get("checkpoint_interval", 50),
        train_sample_frac=TRAIN_SAMPLE_FRAC,
    )
    train_dir = get_training_dir(CASE_STUDY_ID, spec)

    # Skip if this config's hash is already complete (unless FORCE_RETRAIN)
    _status = training_run_status(CASE_STUDY_ID, spec)
    _training_hash = training_hash_from_spec(spec)
    _split_rows = load_prediction_sets(
        CASE_STUDY_ID,
        training_hash=_training_hash,
        split=PREDICTION_SPLIT,
    )
    _split_complete = not _split_rows.is_empty()
    if _status.complete and _split_complete and not FORCE_RETRAIN:
        # Already trained + registered: rebuild a minimal result from the
        # registry so the grid + learning-curve sections render on a
        # fully-cached checkout. (A bare `continue` here drops the config
        # from `results`, printing an empty grid when every config is
        # registered.) best_ic is the authoritative registered value;
        # best_iter and the curves come from learning_curves.parquet.
        _pred_hash = _split_rows["prediction_hash"][0]
        _metrics = load_prediction_metrics(CASE_STUDY_ID, prediction_hash=_pred_hash)
        _best_ic = float(_metrics["ic_mean"][0]) if not _metrics.is_empty() else float("nan")
        _curves = []
        _lc_path = train_dir / "learning_curves.parquet"
        if _lc_path.exists():
            _curves = pl.read_parquet(_lc_path).to_dicts()
        _best_iter = 0
        if _curves:
            _best_iter = int(max(_curves, key=lambda c: c["ic_mean"])["iteration"])
        print(
            f"  {cfg['config_name']:25s}  iter={_best_iter:4d}  IC={_best_ic:+.4f}  "
            f"(cached, {_status.summary()})"
        )
        results.append(
            {
                "config_name": cfg["config_name"],
                "best_iter": _best_iter,
                "best_ic": _best_ic,
                "elapsed_s": 0.0,
                "learning_curves": _curves,
                "cached": True,
            }
        )
        continue
    if _status.complete and not _split_complete:
        print(f"  {cfg['config_name']:25s}  RETRAIN — missing {PREDICTION_SPLIT} predictions")
    elif _status.partial:
        print(f"  {cfg['config_name']:25s}  RETRAIN — partial state: {_status.summary()}")

    result = train_gbm_config(
        cfg,
        fold_data,
        feature_names=feature_names,
        device=DEVICE,
        max_bin=MAX_BIN,
        entity_col=entity_col,
        date_col=date_col,
        task_type=mds.task_type,
        class_values=mds.class_values,
        save_dir=train_dir,
    )
    results.append(result)
    print(
        f"  {result['config_name']:25s}  iter={result['best_iter']:4d}  "
        f"IC={result['best_ic']:+.4f}  ({result['elapsed_s']:.0f}s)"
    )

    # Register immediately after training — incremental save protects against
    # interruption losing work on large sweeps.
    register_gbm_result(
        CASE_STUDY_ID,
        result,
        cfg,
        label_col,
        n_folds=len(fold_data),
        max_bin=MAX_BIN,
        entry_point="07_gbm",
        date_col=date_col,
        entity_col=entity_col,
        train_sample_frac=TRAIN_SAMPLE_FRAC,
        prediction_split=PREDICTION_SPLIT,
    )

  leaves_63_mae              iter= 250  IC=+0.0054  (cached, complete (hash=b3bafe43fe32))
  leaves_7_mae               iter= 500  IC=+0.0060  (cached, complete (hash=bb6c51aa40fd))
  leaves_31_mae              iter= 150  IC=+0.0024  (cached, complete (hash=c0f0f2c2f335))
  leaves_31_huber            iter= 150  IC=+0.0038  (cached, complete (hash=404a3a26a2dd))
  leaves_15_huber            iter= 100  IC=+0.0037  (cached, complete (hash=e842da6c9d6e))
  leaves_63_huber            iter= 500  IC=+0.0047  (cached, complete (hash=33c756b47841))
  leaves_15_mae              iter= 250  IC=+0.0030  (cached, complete (hash=4753395c6900))
  default_mae                iter= 250  IC=+0.0036  (cached, complete (hash=c368437f7d57))
  leaves_7_huber             iter= 500  IC=+0.0022  (cached, complete (hash=5b68ae182f9d))
  default_huber              iter= 400  IC=+0.0050  (cached, complete (hash=3e0900b300cc))
  default_mse                iter= 400  IC=+0.0051  (cached, complete (hash=fd2dc33c9405))

## 4. Grid Results

All configs ranked by peak IC (best checkpoint). The best config × iteration
combination is selected for downstream prediction generation.

In [8]:
results.sort(key=lambda r: r["best_ic"], reverse=True)
best = results[0] if results else None

print(f"{'Config':25s}  {'Iter':>5s}  {'IC':>8s}  {'Time':>6s}")
print("-" * 50)
for r in results:
    marker = " *" if r is best else ""
    print(
        f"  {r['config_name']:25s}  {r['best_iter']:5d}  {r['best_ic']:+.4f}  {r['elapsed_s']:5.0f}s{marker}"
    )

if best:
    print(f"\nBest: {best['config_name']} @ {best['best_iter']} trees (IC={best['best_ic']:+.4f})")

Config                      Iter        IC    Time
--------------------------------------------------
  leaves_7_mae                 500  +0.0060      0s *
  leaves_63_mae                250  +0.0054      0s
  default_mse                  400  +0.0051      0s
  default_huber                400  +0.0050      0s
  leaves_63_mse                500  +0.0047      0s
  leaves_63_huber              500  +0.0047      0s
  leaves_31_huber              150  +0.0038      0s
  leaves_31_mse                150  +0.0038      0s
  leaves_15_huber              100  +0.0037      0s
  leaves_15_mse                100  +0.0037      0s
  default_mae                  250  +0.0036      0s
  leaves_15_mae                250  +0.0030      0s
  leaves_31_mae                150  +0.0024      0s
  leaves_7_huber               500  +0.0022      0s
  leaves_7_mse                 500  +0.0022      0s

Best: leaves_7_mae @ 500 trees (IC=+0.0060)


## 5. Learning Curves

IC at checkpoints (every 50 iterations) for each config. Configs that peak
early and decay indicate overfitting; those that plateau show good regularization.

In [9]:
all_curves = pl.DataFrame([c for r in results for c in r["learning_curves"]])
if all_curves.height > 0:
    checkpoints = sorted(all_curves["iteration"].unique().to_list())
    display_cps = [cp for cp in [50, 100, 200, 300, 500] if cp in checkpoints]

    print(f"{'Config':25s}", end="")
    for cp in display_cps:
        print(f" {cp:>7d}", end="")
    print()

    for r in results:
        cfg_data = all_curves.filter(pl.col("config") == r["config_name"])
        print(f"  {r['config_name']:25s}", end="")
        for cp in display_cps:
            row = cfg_data.filter(pl.col("iteration") == cp)
            if row.height > 0:
                print(f" {row['ic_mean'][0]:+7.4f}", end="")
            else:
                print(f" {'N/A':>7s}", end="")
        print()

Config                         50     100     200     300     500
  leaves_7_mae              +0.0011 +0.0013 +0.0019 +0.0040 +0.0060
  leaves_63_mae             +0.0032 +0.0033 +0.0045 +0.0028 +0.0027
  default_mse               +0.0003 +0.0016 +0.0049 +0.0042 +0.0044
  default_huber             +0.0003 +0.0017 +0.0048 +0.0042 +0.0044
  leaves_63_mse             +0.0034 +0.0033 +0.0045 +0.0043 +0.0047
  leaves_63_huber           +0.0034 +0.0033 +0.0045 +0.0043 +0.0047
  leaves_31_huber           +0.0009 +0.0037 +0.0034 +0.0030 +0.0034
  leaves_31_mse             +0.0009 +0.0037 +0.0034 +0.0030 +0.0034
  leaves_15_huber           +0.0030 +0.0037 +0.0014 +0.0010 +0.0014
  leaves_15_mse             +0.0030 +0.0037 +0.0014 +0.0010 +0.0014
  default_mae               +0.0023 +0.0017 +0.0020 +0.0032 +0.0035
  leaves_15_mae             +0.0012 +0.0019 +0.0027 +0.0030 +0.0016
  leaves_31_mae             +0.0025 +0.0030 +0.0026 +0.0027 +0.0023
  leaves_7_huber            +0.0013 +0.0012 +0.001

## 6. Registration Complete

Each config was registered immediately after training (see Section 3).
This protects against interruption — all completed configs are already
persisted in `run_log/registry.db`.

In [10]:
print(f"All {len(results)} configs registered.")

All 15 configs registered.


## 7. Key Takeaways

The highest GBM IC on fwd_ret_15m is +0.0060 (leaves_7_mae, HAC CI [+0.0026,
+0.0094], excludes zero) — modestly above linear's +0.0049 (ridge_a1000000.0,
CI [+0.0022, +0.0076]). The gap is small relative to other case studies where
GBM typically doubles the linear IC: at 15-minute frequency microstructure
signals are largely linear (signed flow predicts short-term price movement)
and tree-based models add only marginal non-linearity. The horizon effect is
the more striking result — the highest GBM IC moves +0.0043 → +0.0060 → +0.0104
across 60m → 15m → 5m, with both 5m and 15m CIs excluding zero and the 60m CI
straddling. Small-leaf MAE GBM achieves the highest IC at every horizon;
capacity for interactions does not reward at this frequency.